# Join Simon's model fits to the existing table for the ultimate concentration of plotting power!

This should basically be a lesser version of my mega_table_joining.ipynb file in that it will be joining tables that are far smaller. I only need to join on Simon's parameters for the DCs and the DQs. Although... I could probably also do WD J1644-0449 and WD J2356-209 as well since I've got those from Simon separately. I'm not going to do the WD+dM parameters since I only have those for WISEA J0615-1247.

In [1]:
from __future__ import print_function
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from astropy.io import fits
from glob import glob
from astropy.time import Time
from astropy import coordinates as coord
from astropy import units as u
from astropy import constants as const
from astropy import convolution as conv
from astropy.table import Table, Column, vstack, join
#import scipy.interpolate as scinterp
import time
import pyvo
#from astroquery.vizier import Vizier

#from astroquery.gaia import Gaia


sys.path.append('../')



In [2]:
input_file='full_MORDOR_survey_1681141339_fullphotometry.csv'
dq_file='MORDOR_results_DQ_Blouin_fits.csv'
dc_file='MORDOR_results_DC_Blouin_fits.csv'
dz_file='MORDOR_results_DZ_Blouin_fits.csv'
input_directory='/Users/BenKaiser/Desktop/MORDOR_Survey_paper/MORDOR_Survey_forPaper/Goodman_spectra/'

In [3]:
name_base=input_file.split('.')[0]
name_base=name_base+'_fitparams'
output_filename=name_base+'.csv'

Probably want to clean up the naming convention of the columns here... before sticking them into my table. Also need to lower the names to be all lowercase because that's how I format stuff. I also probably want this to match the naming convention I've been using for all of my abundance analysis for the last several years

In [4]:
input_table=Table.read(input_directory+input_file,format='ascii.csv')
dq_table=Table.read(input_directory+dq_file,format='ascii.csv')
dc_table=Table.read(input_directory+dc_file,format='ascii.csv')
dz_table=Table.read(input_directory+dz_file,format='ascii.csv')

In [5]:
dq_table.pprint()

   dr3_source_id    teff teff_err  logg logg_err c/he c/he_err
------------------- ---- -------- ----- -------- ---- --------
5140433498003756160 4402       96  8.12    0.097 -7.0      0.2
4673772519470806144 5380       98  8.47    0.077 -5.1      0.1
6596688068617463808 5088      166 8.047    0.157 -6.2      0.1
1761085712628042112 5209      198 8.293    0.156 -5.4      0.2


I cleaned up the DC table manually since there were so few that had mixed atmospheres as best fits it turns out. 

Really I should be joining the DZ file/table into the greater table first because it includes the columns that the other ones have, so I'm pretty sure I can do a different kind of join. Or wait, I can do that add_index function and then just index using whatever column to add in the columns that are needed from the new table.

In [6]:
new_table=join(input_table,dz_table,join_type='left',keys='name')

In [7]:
print(new_table['note'].dtype)

<U4


In [8]:
new_table['c/he'].dtype=np.float64
new_table['c/he_err'].dtype=np.float64
new_table['note']=new_table['note'].astype(np.dtype('S200'))

In [9]:
input_table.pprint()

       name                 designation           ... ukidss_pmde ukidss_cl
----------------- ------------------------------- ... ----------- ---------
MORDOR J0325+1236   b'Gaia DR2 17497872857713920' ...          --        --
MORDOR J0349+1241   b'Gaia DR2 37101100029189376' ...          --        --
  LSPM J0253+2135  b'Gaia DR2 109336535778050944' ...          --        --
  SDSS J0447+2430  b'Gaia DR2 147065972342145152' ...          --        --
MORDOR J0447+8026  b'Gaia DR2 556697752349562112' ...          --        --
MORDOR J0244+8145  b'Gaia DR2 568855743906926976' ...          --        --
  ULAS J0815+2427  b'Gaia DR2 679275191263785984' ...       -56.3        -1
  SDSS J0824+2705  b'Gaia DR2 683016588814929280' ...          --        -1
  SDSS J1040+3936  b'Gaia DR2 779233790504599168' ...          --        --
  SDSS J1125+4453  b'Gaia DR2 784551196240140928' ...          --        --
              ...                             ... ...         ...       ...
MORDOR J1534

In [10]:
new_table.pprint()

       name                 designation           ... log_q note
----------------- ------------------------------- ... ----- ----
       LEHPM 1774 b'Gaia DR2 5140433498003756160' ...    --   --
        LP 178-49 b'Gaia DR2 1411958889265082496' ...    --   --
  LSPM J0211+0835 b'Gaia DR2 2521858084423378176' ...    --   --
  LSPM J0253+2135  b'Gaia DR2 109336535778050944' ...    --   --
MORDOR J0008-2404 b'Gaia DR2 2336521827465439616' ...    --   --
MORDOR J0110-5600 b'Gaia DR2 4913565903725184640' ...    --   --
MORDOR J0110-5926 b'Gaia DR2 4909222871450620416' ...    --   --
MORDOR J0126-4707 b'Gaia DR2 4930805868092016256' ...    --   --
MORDOR J0135-6306 b'Gaia DR2 4712157368044092928' ...    --   --
MORDOR J0225-3312 b'Gaia DR2 4967454759604815360' ...    --   --
              ...                             ... ...   ...  ...
  SDSS J1536+1718 b'Gaia DR2 1197245124720367104' ...    --   --
  SDSS J1555+1330 b'Gaia DR2 1191644246850711040' ...    --   --
  SDSS J1617+4944 b'Gaia 

We don't want to do a join for the other tables I'm pretty sure because we already have all of the columns that it would be populating. But we want to retain all 120 columns as well. I'm not certain that there is a method of joining that would work that way, so I'm going to try doing it with indexing add_index and .loc[]

In [11]:
new_table.add_index('dr3_source_id')

In [12]:
for colname in dc_table.colnames[1:]:
    try:
        print(new_table.loc[dc_table['dr3_source_id']][colname],dc_table[colname])
        new_table.loc[dc_table['dr3_source_id']][colname]=dc_table[colname]
    except KeyError as error:
        print("KeyError:",error)

teff
----
  --
  --
  --
  --
  --
  --
  --
  --
  --
  --
 ...
  --
  --
  --
  --
  --
  --
  --
  --
  --
  --
  --
Length = 53 rows teff
----
5317
5226
5179
5142
5042
5039
5021
4939
4777
4753
 ...
3782
3655
3587
3514
3500
3478
3465
3447
3423
3344
3303
Length = 53 rows
teff_err
--------
      --
      --
      --
      --
      --
      --
      --
      --
      --
      --
     ...
      --
      --
      --
      --
      --
      --
      --
      --
      --
      --
      --
Length = 53 rows teff_err
--------
     191
     307
      52
      51
     238
      70
      37
     143
     180
      31
     ...
     473
      26
      88
     125
     150
      74
     125
    1000
     116
    1000
      34
Length = 53 rows
logg
----
  --
  --
  --
  --
  --
  --
  --
  --
  --
  --
 ...
  --
  --
  --
  --
  --
  --
  --
  --
  --
  --
  --
Length = 53 rows  logg
-----
 8.15
8.504
7.979
7.783
8.065
8.012
7.971
7.412
7.856
7.836
  ...
  7.0
7.476
  7.0
7.649
  7.0
7.634
7.464
7.2

In [13]:
for row in new_table:
    print(row['h/he'])

--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
--
-2.0
-1.5
--
--


    Take a table (the DC table definitely) and fix the parts of columns that can't actually be used with another
    table. The H/He column stands out with this problem. I need to convert it to my notation for uncertainties
    instead of the present <7... I could probably just do this manually in the CSV since there are so few...
    
    Ah wait... those are for the surface gravity... uh oh.
    
    I implemented the uncertainty as -100 for log(g) just like I did for the abundances to indicate an upper limit.

In [14]:
def weave_table(feeder_table,eater_table):
    """
    feeder_table is the table you want to pull values from
    
    eater_table is the table you are weaving the values into
    
    The eater_table should already have the dr3_source_id add_index done to it, so that can be used in here
    
    """
    for row in feeder_table:
        #print(eater_table.loc[row['dr3_source_id']])
        #print(row)
        eater_table.loc[row['dr3_source_id']][feeder_table.colnames[1:]]=row[feeder_table.colnames[1:]]
    return eater_table

In [15]:
new_table=weave_table(dc_table,new_table)

In [16]:
new_table=weave_table(dq_table,new_table)

In [17]:
new_table.pprint()
for row in new_table:
    print(row['name'],row['cr/he'],row['cr/he_err'])

       name                 designation           ... log_q note
----------------- ------------------------------- ... ----- ----
       LEHPM 1774 b'Gaia DR2 5140433498003756160' ...    --   --
        LP 178-49 b'Gaia DR2 1411958889265082496' ...    --   --
  LSPM J0211+0835 b'Gaia DR2 2521858084423378176' ...    --   --
  LSPM J0253+2135  b'Gaia DR2 109336535778050944' ...    --   --
MORDOR J0008-2404 b'Gaia DR2 2336521827465439616' ...    --   --
MORDOR J0110-5600 b'Gaia DR2 4913565903725184640' ...    --   --
MORDOR J0110-5926 b'Gaia DR2 4909222871450620416' ...    --   --
MORDOR J0126-4707 b'Gaia DR2 4930805868092016256' ...    --   --
MORDOR J0135-6306 b'Gaia DR2 4712157368044092928' ...    --   --
MORDOR J0225-3312 b'Gaia DR2 4967454759604815360' ...    --   --
              ...                             ... ...   ...  ...
  SDSS J1536+1718 b'Gaia DR2 1197245124720367104' ...    --   --
  SDSS J1555+1330 b'Gaia DR2 1191644246850711040' ...    --   --
  SDSS J1617+4944 b'Gaia 

In [18]:
dc_table[dc_table.colnames[1:]]

teff,teff_err,logg,logg_err,h_or_he,h/he,h/he_err,note
int64,int64,float64,float64,str3,float64,int64,str178
5317,191,8.15,0.141,H,--,--,there is a feature in the spectrum at the H-alpha wavelength that is strong enough to hide the real H-alpha
5226,307,8.504,0.176,H,--,--,there is a feature in the spectrum at the H-alpha wavelength that is strong enough to hide the real H-alpha
5179,52,7.979,0.149,H,--,--,there is a feature in the spectrum at the H-alpha wavelength that is strong enough to hide the real H-alpha
5142,51,7.783,0.102,H,--,--,I would call this a DA
5042,238,8.065,0.185,H,--,--,spectrum noisy enough that H-alpha is plausibly just hidden in the noise
5039,70,8.012,0.13,mix,-5.0,--,pure H ruled out from spectroscopy; low-quality fit (photometry looks inconsistent?); assumed small H trace
5021,37,7.971,0.096,H,--,--,spectrum noisy enough that H-alpha is plausibly just hidden in the noise
4939,143,7.412,0.171,H,--,--,there is a feature in the spectrum at the H-alpha wavelength that is strong enough to hide the real H-alpha
4777,180,7.856,0.187,mix,-5.0,--,pure H ruled out from spectroscopy; assumed small H trace


In [19]:
new_table.loc[dc_table['dr3_source_id']]

name,designation,dr2_source_id,gf19,pwd,f_pwd,sp_type,sed_sp_type,dr3_source_id,magnitude_difference,solution_id,DESIGNATION,SOURCE_ID,random_index,ref_epoch,ra,ra_error,dec,dec_error,parallax,parallax_error,parallax_over_error,pm,pmra,pmra_error,pmdec,pmdec_error,ra_dec_corr,ra_parallax_corr,ra_pmra_corr,ra_pmdec_corr,dec_parallax_corr,dec_pmra_corr,dec_pmdec_corr,parallax_pmra_corr,parallax_pmdec_corr,pmra_pmdec_corr,astrometric_n_obs_al,astrometric_n_obs_ac,astrometric_n_good_obs_al,astrometric_n_bad_obs_al,astrometric_gof_al,astrometric_chi2_al,astrometric_excess_noise,astrometric_excess_noise_sig,astrometric_params_solved,astrometric_primary_flag,nu_eff_used_in_astrometry,pseudocolour,pseudocolour_error,ra_pseudocolour_corr,dec_pseudocolour_corr,parallax_pseudocolour_corr,pmra_pseudocolour_corr,pmdec_pseudocolour_corr,astrometric_matched_transits,visibility_periods_used,astrometric_sigma5d_max,matched_transits,new_matched_transits,matched_transits_removed,ipd_gof_harmonic_amplitude,ipd_gof_harmonic_phase,ipd_frac_multi_peak,ipd_frac_odd_win,ruwe,scan_direction_strength_k1,scan_direction_strength_k2,scan_direction_strength_k3,scan_direction_strength_k4,scan_direction_mean_k1,scan_direction_mean_k2,scan_direction_mean_k3,scan_direction_mean_k4,duplicated_source,phot_g_n_obs,phot_g_mean_flux,phot_g_mean_flux_error,phot_g_mean_flux_over_error,phot_g_mean_mag,phot_bp_n_obs,phot_bp_mean_flux,phot_bp_mean_flux_error,phot_bp_mean_flux_over_error,phot_bp_mean_mag,phot_rp_n_obs,phot_rp_mean_flux,phot_rp_mean_flux_error,phot_rp_mean_flux_over_error,phot_rp_mean_mag,phot_bp_rp_excess_factor,phot_bp_n_contaminated_transits,phot_bp_n_blended_transits,phot_rp_n_contaminated_transits,phot_rp_n_blended_transits,phot_proc_mode,bp_rp,bp_g,g_rp,radial_velocity,radial_velocity_error,rv_method_used,rv_nb_transits,rv_nb_deblended_transits,rv_visibility_periods_used,rv_expected_sig_to_noise,rv_renormalised_gof,rv_chisq_pvalue,rv_time_duration,rv_amplitude_robust,rv_template_teff,rv_template_logg,rv_template_fe_h,rv_atm_param_origin,vbroad,vbroad_error,vbroad_nb_transits,grvs_mag,grvs_mag_error,grvs_mag_nb_transits,rvs_spec_sig_to_noise,phot_variable_flag,l,b,ecl_lon,ecl_lat,in_qso_candidates,in_galaxy_candidates,non_single_star,has_xp_continuous,has_xp_sampled,has_rvs,has_epoch_photometry,has_epoch_rv,has_mcmc_gspphot,has_mcmc_msc,in_andromeda_survey,classprob_dsc_combmod_quasar,classprob_dsc_combmod_galaxy,classprob_dsc_combmod_star,teff_gspphot,teff_gspphot_lower,teff_gspphot_upper,logg_gspphot,logg_gspphot_lower,logg_gspphot_upper,mh_gspphot,mh_gspphot_lower,mh_gspphot_upper,distance_gspphot,distance_gspphot_lower,distance_gspphot_upper,azero_gspphot,azero_gspphot_lower,azero_gspphot_upper,ag_gspphot,ag_gspphot_lower,ag_gspphot_upper,ebpminrp_gspphot,ebpminrp_gspphot_lower,ebpminrp_gspphot_upper,libname_gspphot,clean_panstarrs1_oid,ps1_original_ext_source_id,ps1_angular_distance,ps1_number_of_neighbours,ps1_number_of_mates,ps1_xm_flag,ps1_obj_name,ps1_obj_id,ps1_ra,ps1_dec,ps1_ra_error,ps1_dec_error,ps1_epoch_mean,ps1_g_mean_psf_mag,ps1_g_mean_psf_mag_error,ps1_g_flags,ps1_r_mean_psf_mag,ps1_r_mean_psf_mag_error,ps1_r_flags,ps1_i_mean_psf_mag,ps1_i_mean_psf_mag_error,ps1_i_flags,ps1_z_mean_psf_mag,ps1_z_mean_psf_mag_error,ps1_z_flags,ps1_y_mean_psf_mag,ps1_y_mean_psf_mag_error,ps1_y_flags,ps1_n_detections,ps1_zone_id,ps1_obj_info_flag,ps1_quality_flag,tmass_original_ext_source_id,tmass_angular_distance,tmass_xm_flag,tmass_clean_tmass_psc_xsc_oid,tmass_number_of_neighbours,tmass_number_of_mates,tmass_ph_qual,tmass_tmass_oid,tmass_designation,tmass_ra,tmass_dec,tmass_err_maj,tmass_err_min,tmass_err_ang,tmass_j_mag,tmass_j_mag_error,tmass_h_mag,tmass_h_mag_error,tmass_ks_mag,tmass_ks_mag_error,tmass_ext_key,tmass_j_date,sdss_original_ext_source_id,sdss_angular_distance,sdss_gaia_astrometric_params,sdss_sdssdr9_oid,sdss_number_of_neighbours,sdss_number_of_mates,sdss_best_neighbour_multiplicity,sdss_obj_id,sdss_thing_id,sdss_ra,sdss_dec,sdss_ra_er

Ok, and then I want this to be sorted by RA in the saved file because that's how this is usually done.

In [21]:
sorted_order=np.argsort(new_table['ra'])
sorted_table=new_table[sorted_order]

In [22]:
for row in sorted_table:
    print(row['ra'],row['name'],row['sp_type'])

2.1239407564538393 MORDOR J0008-2404 DC
12.371270083832894 SDSS J0049+2244 DC
17.509516572757104 MORDOR J0110-5926 DC
17.551231306942856 MORDOR J0110-5600 WDdM
21.693921932358695 MORDOR J0126-4707 WDdM
23.03158504644024 SDSS J0132+1823 WDdM
23.89987243809605 MORDOR J0135-6306 DC
25.188378089676405 LEHPM 1774 DQpec
32.90126629650901 LSPM J0211+0835 DC
36.37789734876497 MORDOR J0225-3312 DC
41.223687976863935 MORDOR J0244+8145 --
42.00239206050494 MORDOR J0248-1556 DC
43.26859378854847 MORDOR J0253-3334 ??
43.3744685582577 LSPM J0253+2135 DC
49.75447303223039 MORDOR J0319-7359 --
51.26664139439566 MORDOR J0325+1236 DC
52.4378271421557 MORDOR J0329-4306 ??
53.58095311786865 MORDOR J0334-6348 DQpec
55.442904404583516 MORDOR J0341-1029 DC
57.41360630980917 MORDOR J0349+1241 WDdM
60.0009799054693 MORDOR J0400-2041 DC
64.91747421132247 MORDOR J0419-0745 WDdM
66.09287297776248 MORDOR J0424-2001 WDdM
67.96844222544722 MORDOR J0431+1502 DC
68.31847222254255 MORDOR J0433-4252 DC
69.31662758499185

In [23]:
sorted_table.write(input_directory+output_filename,format='ascii.csv')

This seems to be completely done correctly. So now I need to make another notebook to calculate the masses and uncertainties for each white dwarf